In [1]:
import pandas as pd
df=pd.read_csv(r"C:\Users\HP\Downloads\Recommendation System\Recommendation System\anime.csv")
df.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [2]:
df.shape

(12294, 7)

In [3]:
df.isnull().sum()

anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64

In [4]:
df['genre'] = df['genre'].fillna('Unknown')
df['type'] = df['type'].fillna('Unknown')

# Ratings are numerical; we fill missing with the median to avoid skewing
df['rating'] = df['rating'].fillna(df['rating'].median())
df.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [5]:
df.isnull().sum()

anime_id    0
name        0
genre       0
type        0
episodes    0
rating      0
members     0
dtype: int64

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler

# 1. Vectorize Genres
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['genre'])

# 2. Normalize Ratings
scaler = MinMaxScaler()
df['scaled_rating'] = scaler.fit_transform(df[['rating']])

# 3. Combine Features
# We convert the sparse matrix to a dense array and add the rating column
from scipy.sparse import hstack
final_features = hstack([tfidf_matrix, df[['scaled_rating']].values])
print(final_features)

<COOrdinate sparse matrix of dtype 'float64'
	with 52773 stored elements and shape (12294, 48)>
  Coords	Values
  (0, 8)	0.44024715174602147
  (0, 30)	0.4903869890743284
  (0, 32)	0.5189547975465538
  (0, 41)	0.5444161684130491
  (1, 8)	0.33583365500987794
  (1, 0)	0.29464923376142327
  (1, 1)	0.3176066463834324
  (1, 10)	0.3196092910272099
  (1, 20)	0.44963166999178716
  (1, 23)	0.5215484702178532
  (1, 36)	0.35098726333369934
  (2, 0)	0.2506314436398565
  (2, 36)	0.29855310799740736
  (2, 5)	0.20076598368966767
  (2, 15)	0.37886802211807874
  (2, 26)	0.4480162264049301
  (2, 31)	0.5507572919931326
  (2, 33)	0.28297511283487664
  (2, 11)	0.28297511283487664
  (3, 33)	0.3904035047060254
  (3, 11)	0.3904035047060254
  (3, 42)	0.8337686771680168
  (4, 0)	0.2506314436398565
  (4, 36)	0.29855310799740736
  (4, 5)	0.20076598368966767
  :	:
  (12269, 47)	0.2893157262905162
  (12270, 47)	0.15966386554621853
  (12271, 47)	0.4237695078031212
  (12272, 47)	0.29651860744297714
  (12273, 47)	0.279

In [7]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute Similarity Matrix (Note: For very large datasets, compute this on-the-fly)
cosine_sim = cosine_similarity(final_features, final_features)

def recommend_anime(title, threshold=0.5):
    try:
        # Get index of the target anime
        idx = df[df['name'] == title].index[0]
        
        # Get similarity scores for all anime
        sim_scores = list(enumerate(cosine_sim[idx]))
        
        # Filter based on threshold and sort
        # x[1] is the similarity score
        filtered_scores = [x for x in sim_scores if x[1] >= threshold]
        sorted_scores = sorted(filtered_scores, key=lambda x: x[1], reverse=True)
        
        # Get top 10 (excluding the anime itself)
        top_indices = [i[0] for i in sorted_scores[1:11]]
        
        return df['name'].iloc[top_indices]
    except IndexError:
        return "Anime not found in dataset."

print(recommend_anime("Naruto", threshold=0.4))

615                                    Naruto: Shippuuden
1103    Boruto: Naruto the Movie - Naruto ga Hokage ni...
486                              Boruto: Naruto the Movie
1343                                          Naruto x UT
1472          Naruto: Shippuuden Movie 4 - The Lost Tower
1573    Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...
2458                 Naruto Shippuuden: Sunny Side Battle
2997    Naruto Soyokazeden Movie: Naruto to Mashin to ...
784            Naruto: Shippuuden Movie 6 - Road to Ninja
1796                                       Rekka no Honoo
Name: name, dtype: object


In [8]:
df_ratings=df['scaled_rating']
print(df_ratings)

0        0.924370
1        0.911164
2        0.909964
3        0.900360
4        0.899160
           ...   
12289    0.297719
12290    0.313325
12291    0.385354
12292    0.397359
12293    0.454982
Name: scaled_rating, Length: 12294, dtype: float64


In [14]:
from sklearn.model_selection import train_test_split

# Split the rows
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Reset indices so we can find them easily later
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

In [16]:
def get_metrics(test_df, train_df, k=10):
    # Vectorize genres
    from sklearn.feature_extraction.text import CountVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    
    cv = CountVectorizer(stop_words='english')
    train_matrix = cv.fit_transform(train_df['genre'].fillna(''))
    test_matrix = cv.transform(test_df['genre'].fillna(''))
    
    # Compare Test Anime to Train Anime
    sim_matrix = cosine_similarity(test_matrix, train_matrix)
    
    total_p, total_r = 0, 0
    
    for i in range(len(test_df)):
        # Get Top K matches from the training library
        scores = list(enumerate(sim_matrix[i]))
        top_k = sorted(scores, key=lambda x: x[1], reverse=True)[:k]
        
        # Calculate Hits
        test_genres = set(test_df.iloc[i]['genre'].split(','))
        hits = 0
        for idx, score in top_k:
            rec_genres = set(train_df.iloc[idx]['genre'].split(','))
            if len(test_genres & rec_genres) / len(test_genres) >= 0.75:
                hits += 1
        
        # Precision: Hits out of K recommendations
        precision = hits / k
        # Recall: Hits out of all possible matches in the library
        # (Simplified: we use 1 if any hits exist, 0 otherwise for a basic check)
        recall = 1 if hits > 0 else 0 
        
        total_p += precision
        total_r += recall

    avg_p = total_p / len(test_df)
    avg_r = total_r / len(test_df)
    f1 = 2 * (avg_p * avg_r) / (avg_p + avg_r) if (avg_p + avg_r) > 0 else 0
    
    return avg_p, avg_r, f1

precision, recall, f1 = get_metrics(test_df, train_df)
print(f"Precision: {precision:.2f}, Recall: {recall:.2f}, F1: {f1:.2f}")

Precision: 0.83, Recall: 0.97, F1: 0.89
